# PerfumeInsightLab
## Notebook 03 : Quantitative EDA
--------------------------------------------------------------------  
#### Part of the multi-notebook EDA workflow : 01 (Data Overview) → 02 (Thematic EDA) → 03 → 04 (Storytelling & Insights)
#### 1. Import librairies
#### 2. Load dataset
#### 3. Feature engineering
#### 4. Data quality verification
#### 5. Exploration & Analysis
-  TOP BRANDS  
-  RATINGS & POPULARITY  
-  YEAR / TEMPORAL ANALYSIS   
-  GENDER ANALYSIS  
-  CORRELATIONS  

In [ ]:
# ------------------------------------------------------------------
# 1. Import libraries
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import unidecode

# ------------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------------
df = pd.read_csv("../data/fra_cleaned_v2.csv", encoding="ISO-8859-1", sep=";")

In [ ]:
# ------------------------------------------------------------------
# 3. Feature engineering
# ------------------------------------------------------------------
# Weighted Rating calculation (corrected metric)

df_rated = df.dropna(subset=['Rating Value', 'Rating Count']).copy()      # Filter data to calculate the mean (C) and the threshold (m) accurately

                                                                          # Define the parameters for the weighted rating formula :
C = df_rated['Rating Value'].mean()            # C = Mean rating across the entire rated sample (N=509)
m = df_rated['Rating Count'].quantile(0.90)    # m = Minimum number of votes required (using the 90th percentile of Rating Count)
m = max(100, m)                                # Ensure 'm' is at least 100 for credibility

print(f"GLOBAL MEAN RATING (C) : {C:.2f}")
print(f"MINIMUM VOTE THRESHOLD (m) : {m:.0f}")
# print(f"Weighted Rating calculated. Global C: {C:.2f}, Threshold m: {m:.0f}")

def weighted_rating(row, m=m, C=C):                                       # Define the Weighted Rating function
    v = row['Rating Count']
    R = row['Rating Value']
    
    if pd.isna(v) or pd.isna(R):    # Handle NaN values for non-rated perfumes
        return np.nan
     
    return (v*R + m*C) / (v + m)    # Formula: (v*R + m*C) / (v+m)

df['weighted_rating'] = df.apply(weighted_rating, axis=1)                 # Apply the function to create the new column

print("\nTHE 'weighted_rating' COLUMN HAS BEEN CALCULATED ON THE ENTIRE DATAFRAME")
print("THIS COLUMN WILL NOW BE USED FOR ALL SUBSEQUENT PERFORMANCE ANALYSES")

top_weighted = df.sort_values('weighted_rating', ascending=False).head(10).round(3)
print("\nTOP 10 PERFUMES BASED ON WEIGHTED RATING (corrected for popularity bias) :\n")
display(top_weighted[['Perfume', 'Brand', 'Rating Value', 'Rating Count', 'weighted_rating']])

In [ ]:
# ------------------------------------------------------------------
# 4. Data quality verification
# ------------------------------------------------------------------

print("COLUMN TYPES :")              # Checking data types for each column
df.dtypes

In [ ]:
# ------------------------------------------------------------------
# 5. Exploration & Analysis
# ------------------------------------------------------------------

In [ ]:
# -------------------------------------------------------------------- TOP BRANDS --------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------------------------

top_brands = df['Brand'].value_counts().head(10)  
sns.barplot(x=top_brands.values, y=top_brands.index)
plt.title("\nTOP 10 BRANDS BY NUMBER OF PERFUMES\n")
plt.xlabel("Nber of Perfumes")
plt.show()

In [ ]:
df.groupby('Brand')['weighted_rating'].mean().sort_values(ascending=False).head(10).round(2)

In [ ]:
df['Country'].value_counts().head(10)
df.groupby('Brand')['weighted_rating'].mean().sort_values(ascending=False).head(10)
df.groupby(['Country','Brand']).size().reset_index(name='Perfume Count')

In [ ]:
unknown_ratio = df.groupby('Brand')['Perfumer'].apply(
    lambda x: (x == 'unknown').mean() * 100).sort_values(ascending=False)
df['Perfumer'].value_counts().head(10)
df[['Perfumer','Brand']].dropna().drop_duplicates().groupby('Perfumer').nunique()['Brand'].sort_values(ascending=False).head(10)

In [ ]:
# -------------------------------------------------------------- RATINGS & POPULARITYT ---------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------------------------

#### RATINGS & POPULARITY, Data Quality Issue 

The initial Data Overview (EDA_01) revealed a major limitation : the **Rating Value** column contains 23 554 missing values out of a total of 24 063 perfumes (nearly 98% of the dataset).  

All customer performance analyses (Ratings, Popularity, rankings) presented below will be based on the subset of **$N=509$ perfumes** for which a rating and a vote count are available.  
($N$ denotes the size of the sample, which is the exact count of perfumes used for this specific analysis)   
This group of perfumes is inherently the most popular or most discussed.   
The conclusions are a good indicator of best-seller performance, but are not generalizable to the entire market.  

In [ ]:
df_rated = df.dropna(subset=['Rating Value', 'Rating Count']).copy()       # Query to get the sample of rated perfumes (N=509)

print(df_rated.shape)
print(df.columns.tolist())

In [ ]:
# What is the correlation coefficient between Rating Value and Rating Count ?
# What does it imply about the need for a weighted rating system ?

corr = df['Rating Value'].corr(df['Rating Count'])
print(f"CORRELATION BETWEEN RATING VALUE AND RATING COUNT : {corr:.2f}")

if corr > 0.5:
    print("→ STRONG POSITIVE CORRELATION : perfumes with more votes tend to have higher ratings. A weighted rating system could help balance popularity bias.")
elif corr > 0.2:
    print("→ MODERATE POSITIVE CORRELATION : some tendency for popular perfumes to score higher")
else:
    print("→ WEAK CORRELATION : popularity does not strongly affect ratings ; weighted scoring is less necessary")

In [ ]:
min_count = 1000
relevant = df[df['Rating Count'] >= min_count].copy()
top_relevant = relevant.sort_values('weighted_rating', ascending=False)
display(top_relevant[['Perfume','Brand','Year','Rating Value','Rating Count','weighted_rating']].head(10).round(3))

In [ ]:
threshold = df['Rating Count'].quantile(0.90)
print("OVERALL RATING VALUE SUMMARY (simple mean) :\n")
display(df['Rating Value'].describe().round(2))

print("\nOVERALL WEIGHTED RATING SUMMARY :\n")
display(df['weighted_rating'].describe().round(2))

top10pct = df[df['Rating Count'] >= threshold]
print("\nTOP 10% (by Rating Count) RATING VALUE SUMMARY :\n")
display(top10pct['Rating Value'].describe().round(2))

print("\nTOP 10% (by Rating Count) WEIGHTED RATING SUMMARY :\n")
display(top10pct['weighted_rating'].describe().round(2))

plt.hist(df['weighted_rating'].dropna(), bins=20, alpha=0.5, label='All')
plt.hist(top10pct['weighted_rating'].dropna(), bins=20, alpha=0.7, label='Top 10% by Count')
plt.legend()
plt.title('\nWEIGHTED RATING DISTRIBUTION : All vs Top 10% by Rating Count\n')
plt.xlabel('Weighted Rating'); plt.ylabel('Frequency'); plt.show()

In [ ]:
# logarithmic scale to visualize low-frequency data (Top 10% orange bars)
plt.figure(figsize=(10, 6))
plt.hist(df['weighted_rating'].dropna(), bins=50, alpha=0.5, label='All', log=True)
plt.hist(top10pct['weighted_rating'].dropna(), bins=50, alpha=0.7, label='Top 10% by Count', log=True)
plt.legend()
plt.title('WEIGHTED RATING DISTRIBUTION (Log Scale)\n')
plt.xlabel('Weighted Rating')
plt.ylabel('Frequency')
plt.show()

#### Distribution Analysis : All vs. Top 10%

Using a **logarithmic scale** allows us to see the Top 10% most popular perfumes (orange) compared to the rest of the collection (blue).

**Observation :** Most perfumes have a weighted rating very close to **3.99**  
**Stability :** The formula works well because it removes "fake" 5-star ratings from perfumes with only 1 or 2 votes.  
**Consistency :** The most popular perfumes (orange) are all concentrated in the high-score area, proving they are reliable and well-loved by the community.  

The data is now balanced and ready for brand and trend analysis.  

In [ ]:
sns.histplot(df['weighted_rating'], bins=20, kde=True)
plt.title("DISTRIBUTION OF WEIGHTED RATING\n")
plt.show()

#### Weighted Rating Distribution (KDE)

Adding a **Kernel Density Estimate** curve (the blue line) to the distribution provides a smoother view of the weighted ratings.

**Main Peak :** Most perfumes are concentrated around **3.99**. This shows the formula is stable and prevents "fake" high ratings.   
**Low Scores :** The small bumps on the left show groups of perfumes that the community clearly likes less.  
**The "4.00" Limit :** The sharp drop at 4.00 shows it is very difficult to get a perfect score. To go higher, a perfume needs a huge number of positive votes.  

**The weighted system successfully balances the data, making it reliable for further analysis.**

In [ ]:
sns.histplot(df['weighted_rating'].dropna(), bins=10, kde=True)
plt.title("DISTRIBUTION OF WEIGHTED RATING\n")
plt.show()

sns.histplot(df['Rating Count'].dropna(), bins=20, kde=False)
plt.title("DISTRIBUTION OF RATING COUNT\n")
plt.show()

#### Statistical Distribution SummaryThe latest visualizations confirm that the dataset is now stable and ready for analysis.

**Weighted Rating (KDE Chart)**

* **Massive Stability :** The sharp blue peak at **3.99** shows that most perfumes have been successfully balanced by the formula.  
* **Outlier Control :** By pulling scores toward the mean, we eliminated "fake" high ratings from products with very few votes.  
* **Reliability :** This high concentration ensures that any perfume that actually scores above 4.00 or below 3.90 is truly unique and backed by strong community data.  

**Rating Count (Histogram)**

* **Popularity Gap :** The chart shows a classic "long tail" distribution. Most perfumes have a small nber of votes, while a few "superstars" have thousands.  
* **Market Reality :** This justifies why we needed the **Weighted Rating**, we cannot treat a perfume with 10 votes the same as one with 10 000 votes.   
* **Data Readiness :** We now have a clear view of which products drive the most engagement in the dataset.  

In [ ]:
# Are highly reviewed perfumes rated higher ?
sns.scatterplot(x='Rating Value', y='Rating Count', data=df)
plt.title("RATING COUNT vs RATING VALUE (Visualizing the Popularity Bias)\n")
plt.show()

#### Popularity vs. Quality (Scatter Plot)

This chart shows the relationship between the nber of reviews (**Rating Count**) and the score (**Rating Value**).

* **Standardized Scores :** Almost all perfumes are rated exactly **4.00**, whether they have 10 reviews or 30 000.   
* **No Bias :** Popularity does not mean a higher score. A very famous perfume is not necessarily rated better than a niche one.

This confirms that a simple average is not enough to rank perfumes. We need our **Weighted Rating** to find the real winners.

**Note on the Outlier**  
We can see a single, isolated point at Rating 3.0.  
This is an outlier. It represents a perfume that is rated significantly lower than the rest of the dataset.  
While 99% of perfumes are clustered at 4.0, this specific case shows that some fragrances are clearly disliked by the community.  
It proves that our data cleaning didn't "erase" negative feedback ; it just shows that truly low-rated perfumes are very rare in this collection.

In [ ]:
outlier_3 = df[df['Rating Value'] == 3.0]
display(outlier_3)

#### Focus on the 3.0 Outliers

After filtering the data, we identified 5 perfumes with a raw rating of **3.0**  
These perfumes are rare "flops" in a dataset where most products are highly rated.  

How the weighted formula handles the Outliers :
* **The Difference in Votes :** *Western Leather White* has **116 votes**, while the others (like *Little Mermaid*) only have about **26 votes**.
* **The "Safety" Mechanism :** When a perfume has very few votes, the formula doesn't "trust" the 3.0 score yet. It stays cautious and keeps the rating close to the global average (~ 3.95).
* **The "Trust" Factor :** Because *Western Leather White* has many more reviews (116), the formula "believes" the negative feedback more. This is why its weighted score drops further to **3.87**.

The weighted rating protects perfumes from extreme scores when there isn't enough data.   
A perfume needs a **high nber of reviews** to truly prove it is a "flop".  

In [ ]:
# Do rating averages differ by gender ?
df.groupby('Gender')['weighted_rating'].mean().round(2).sort_values(ascending=False)

In [ ]:
# Are certain olfactory families rated higher ?
df.groupby('mainaccord1')['weighted_rating'].mean().sort_values(ascending=False).round(2).head(10)

In [ ]:
# ------------------------------------------------------------ YEAR / TEMPORAL ANALYSIS --------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------------------------

df_year = df[df['Year'] > 1700]['Year'].value_counts().sort_index()
df_year.plot(kind='line', marker='o', figsize=(12,4))
plt.title("PERFUME RELEASES OVER TIME\n")
plt.xlabel("Year")
plt.ylabel("Nber of Perfumes")
plt.show()

In [ ]:
df_year = df.query('Year > 1979')['Year'].value_counts().sort_index()
df_year.plot(kind='line', marker='o')
plt.title('NBER OF PERFUMES RELEASED PER YEAR\n')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

In [ ]:
df_year = df.query("Year > 2009")['Year'].value_counts().sort_index()
df_year.plot(kind='line', marker='o')
plt.title("NBER OF PERFUMES RELEASED PER YEAR\n")
plt.xlabel("Year")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df[df['Year'] > 0]['Year'], bins=50, kde=False)
plt.title('DISTRIBUTION OF PERFUME RELEASE YEARS\n')
plt.xlabel('Year')
plt.ylabel('Nber of Perfumes')
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df[df['Year'] > 1999]['Year'], bins=25, kde=False)
plt.title('DISTRIBUTION OF PERFUME RELEASE YEARS\n')
plt.xlabel('Year')
plt.ylabel('Nber of Perfumes')
plt.show()

In [ ]:
def decade_of(y):
    if y <= 0 or pd.isna(y):
        return 'unknown'
    return f"{(int(y)//10)*10}s"

df['decade'] = df['Year'].apply(decade_of)

valid_decades = ['1990s', '2000s', '2010s', '2020s']
df_filtered = df[df['decade'].isin(valid_decades)].copy()

mainaccords = df_filtered.melt(            # Melt mainaccord columns and count occurrences per decade
    id_vars='decade',
    value_vars=['mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5'],
    value_name='mainaccord'
).dropna(subset=['mainaccord'])

by_decade = (
    mainaccords.groupby(['decade','mainaccord']).size().reset_index(name='count').sort_values(['decade','count'], ascending=[True, False])
)

for d in valid_decades:
    top = by_decade[by_decade['decade'] == d].head(5)
    if not top.empty:
        print(f"\nTOP MAIN ACCORDS IN DECADE {d} :")
        display(top)

In [ ]:
last_year = df['Year'].max()
period_start = last_year - 4                                                            # last 5 calendar years including last_year
recent_df = df[(df['Year'] >= period_start) & (df['Year'] > 0)].copy()

brand_total_recent = recent_df.groupby('Brand').size().reset_index(name='recent_count') # Count releases per brand per year

baseline_start = period_start - 5                                                       # Historical baseline (previous 5 years)
baseline_end = period_start - 1
baseline_df = df[(df['Year'] >= baseline_start) & (df['Year'] <= baseline_end)]
brand_baseline = baseline_df.groupby('Brand').size().reset_index(name='baseline_count')

growth = brand_total_recent.merge(brand_baseline, on='Brand', how='left').fillna(0)      # Merge and compute growth
growth['growth_abs'] = growth['recent_count'] - growth['baseline_count']
growth['growth_pct'] = growth.apply(                                                     # Percentage growth (rounded to 2 decimals)       
    lambda r: round((r['growth_abs'] / r['baseline_count'] * 100), 2) if r['baseline_count'] > 0 else np.nan,
    axis=1
)

print("TOP BRANDS BY ABSOLUTE GROWTH (recent 5y - previous 5y) :")
display(growth.sort_values('growth_abs', ascending=False).head(5))

In [ ]:
print("TOP BRANDS BY PERCENTAGE GROWTH (baseline > 0) :")
display(growth.dropna(subset=['growth_pct']).sort_values('growth_pct', ascending=False).head(5))

In [ ]:
# ----------------------------------------------------------------- GENDER ANALYSIS ------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------------------------

custom_colors = ['#92a8d1',   # Men
                 '#c5b9cd',   # Unisex
                 '#f7cac9']   # Women
                 
order_gender = ['men', 'unisex', 'women']

sns.countplot(x='Gender', data=df, palette=custom_colors, order=order_gender) 
plt.title("NUMBER OF PERFUMES PER GENDER\n")
plt.show()

In [ ]:
recent = df[df['Year'] > 1999]
total_recent = len(recent)
unisex_recent = len(recent[recent['Gender'].str.lower()=='unisex'])
print(f"SHARE OF UNISEX PERFUMES SINCE 2000 : {unisex_recent} / {total_recent} = {100*unisex_recent/total_recent:.2f}%")

In [ ]:
recent = df[df['Year'] > 2009]
total_recent = len(recent)
unisex_recent = len(recent[recent['Gender'].str.lower()=='unisex'])
print(f"SHARE OF UNISEX PERFUMES SINCE 2010 : {unisex_recent} / {total_recent} = {100*unisex_recent/total_recent:.2f}%")

In [ ]:
recent = df[df['Year'] > 2019]
total_recent = len(recent)
unisex_recent = len(recent[recent['Gender'].str.lower()=='unisex'])
print(f"SHARE OF UNISEX PERFUMES SINCE 2020 : {unisex_recent} / {total_recent} = {100*unisex_recent/total_recent:.2f}%")

In [ ]:
# Has the fragrance industry become more unisex in the 21st century ?
custom_colors = ['#92a8d1',   # Men 
                 '#c5b9cd',   # Unisex 
                 '#f7cac9']   # Women 

df[df['Year'] > 1899]\
    .groupby('Year')['Gender'].value_counts(normalize=True).unstack().plot.area(figsize=(12,4), color=custom_colors)

plt.title("EVOLUTION OF PERFUME GENDER DISTRIBUTION\n")
plt.ylabel("Proportion (%)")
plt.xlabel("Year of Release")
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print("TOP 5 PERFUMES WITH HIGHEST WEIGHTED RATINGS PER GENDER :\n")

top_per_gender = (df.groupby('Gender', group_keys=False).apply(lambda x: x.nlargest(5, 'weighted_rating')).reset_index(drop=True))

display(top_per_gender[['Perfume', 'Brand', 'Rating Value', 'Rating Count', 'weighted_rating', 'Gender']])

In [ ]:
# Do women’s perfumes use more floral accords ? Do men’s perfumes favor woody accords ? Which accords dominate unisex perfumes ?
accord_cols = ['mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5']
df.groupby('Gender')[accord_cols].apply(lambda x: x.stack().str.lower().value_counts().head(10))

In [ ]:
# Focus on the 10 most frequent main accords for clearer visualization --> Counts = popularity
# Compares which main accords appear most often for each gender (absolute counts)

top_accords = (
    df[[*accord_cols]].melt(value_name='accord')['accord'].str.lower().value_counts().head(10).index
)

df_top = df[df['mainaccord1'].str.lower().isin(top_accords)]          # Filter the DataFrame to only include these accords

gender_palette = {                # Create a custom palette as dictionary (ensures correct color mapping)
    'men': '#92a8d1',    
    'unisex': '#c5b9cd',  
    'women': '#f7cac9'    
}

plt.figure(figsize=(9,6))
sns.countplot(
    data=df_top, y='mainaccord1', hue='Gender', order=top_accords, palette=gender_palette, linewidth=1, edgecolor='black'
)

plt.title("TOP 10 MAIN ACCORDS BY GENDER\n")
plt.xlabel("Count")
plt.ylabel("Main Accord")
plt.tight_layout()
plt.show()

In [ ]:
# Proportional distribution exploration of the top 10 main accords by gender --> Proportions = preference intensity
# Shows how frequent each accord is relative to all perfumes of that gender (proportional frequency)

accord_counts = (
    df.melt(id_vars='Gender', value_vars=accord_cols, value_name='accord').dropna()
)
top_10 = accord_counts['accord'].str.lower().value_counts().head(10).index

filtered = accord_counts[accord_counts['accord'].str.lower().isin(top_10)]

plt.figure(figsize=(9,6))
sns.histplot(
    data=filtered,
    y='accord',
    hue='Gender',
    multiple='dodge',        # bars side by side
    shrink=0.8,              # spacing between bars
    palette=gender_palette 
)

plt.title("TOP 10 ACCORDS BY GENDER (Proportional Distribution)\n")
plt.xlabel("Frequency")
plt.ylabel("Main Accord")
plt.tight_layout()
plt.show()

In [ ]:
# Olfactory differences according to gender
df.groupby('Gender')[['mainaccord1','mainaccord2','mainaccord3']].apply(lambda x: x.stack().value_counts().head(10))

In [ ]:
# ----------------------------------------------------------------- CORRELATIONS ---------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------------------------

for col in ['Rating Value', 'Rating Count', 'Year']:                               # Convert numeric columns : replace ',' with '.' and cast to float
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float, errors='ignore')

print(df[['Rating Value','Rating Count','Year']].dtypes)                           # Check data types

df_numeric = df[['Rating Value','weighted_rating','Rating Count','Year']].copy()   # Compute and visualize correlation matrix
sns.heatmap(df_numeric.corr(), annot=True, cmap='coolwarm')
plt.title("\nCORRELATION BETWEEN NUMERIC FEATURES (Including WEIGHTED RATING)\n")
plt.show()

#### Correlation Analysis 

This heatmap shows how different numeric values relate to each other.

* **Strong Link (0.78) :** There is a very strong correlation between the Rating Value and the Weighted Rating.  
  This is normal, as the weighted score is built directly from the original rating.
* **The Popularity Insight (0.22) :** There is a small positive link between Rating Count and Weighted Rating.
  This proves that more popular perfumes tend to have slightly better-stabilized scores, but popularity doesn't guarantee a high rating.  
* **No "Old vs. New" Bias (0.059) :** The Year has almost zero correlation with ratings.  
  This means that newer perfumes are not necessarily rated better or worse than older ones.  

The **Weighted Rating** is a reliable metric because it is strongly tied to quality (Rating Value) while still accounting for popularity (Rating Count).

#### Understanding the Correlation Matrix

* **The Diagonal (1.0) :** These red squares represent each variable compared to itself.
* **High Correlation (0.78) :** The strong link between *Rating Value* and *Weighted Rating* proves the formula maintains the original quality sentiment.
* **Low Correlation (Near 0) :** The blue areas show that *Year* (age of perfume) has no impact on the ratings.   
   A new perfume has the same chance of being highly rated as an old classic.
* **Red** indicates a strong positive relationship, while **Blue** indicates no significant statistical link in this dataset.